# MNIST手書き数字分類 - Julia + Flux

このノートブックでは、JuliaとFluxを使用してMNISTデータセットの手書き数字分類モデルを作成し、評価を行います。

## 1. パッケージのインポート

In [ ]:
using Flux
using Flux: onehotbatch, onecold, crossentropy, throttle
using MLDatasets
using Statistics
using Random
using Plots

println("パッケージのインポート完了")

## 2. データセットの読み込み

In [ ]:
# MNISTデータセットの読み込み
train_data = MLDatasets.MNIST(split=:train)
test_data = MLDatasets.MNIST(split=:test)

# データの形状を確認
println("訓練データサイズ: ", size(train_data.features))
println("テストデータサイズ: ", size(test_data.features))
println("訓練ラベル数: ", length(train_data.targets))
println("テストラベル数: ", length(test_data.targets))

## 3. データの前処理

In [ ]:
# データを平坦化してFloat32に変換
function prepare_data(data)
    # 画像を平坦化 (28x28 -> 784)
    X = Float32.(reshape(data.features, 28*28, :))
    # ラベルをone-hot encodingに変換
    y = onehotbatch(data.targets, 0:9)
    return X, y
end

# 訓練データとテストデータの準備
X_train, y_train = prepare_data(train_data)
X_test, y_test = prepare_data(test_data)

println("訓練データ形状: ", size(X_train))
println("訓練ラベル形状: ", size(y_train))
println("テストデータ形状: ", size(X_test))
println("テストラベル形状: ", size(y_test))

## 4. データの可視化

In [ ]:
# 最初の10枚の画像を表示
p = plot(layout=(2, 5), size=(800, 400), legend=false)
for i in 1:10
    img = reshape(X_train[:, i], 28, 28)
    label = onecold(y_train[:, i], 0:9)
    heatmap!(p[i], img, c=:grays, title="Label: $label", aspect_ratio=:equal)
end
display(p)

## 5. モデルの定義

In [ ]:
# ニューラルネットワークモデルの定義
model = Chain(
    Dense(784, 128, relu),    # 入力層 -> 隠れ層1
    Dense(128, 64, relu),     # 隠れ層1 -> 隠れ層2
    Dense(64, 10),            # 隠れ層2 -> 出力層
    softmax                   # 活性化関数
)

println("モデルの定義完了")
println(model)

## 6. 損失関数と最適化アルゴリズムの設定

In [ ]:
# 損失関数
loss(x, y) = crossentropy(model(x), y)

# 最適化アルゴリズム (Adam)
opt = Adam(0.001)

# パラメータの取得
params_model = Flux.params(model)

println("損失関数と最適化アルゴリズムの設定完了")

## 7. 評価関数の定義

In [ ]:
# 精度の計算
function accuracy(x, y)
    pred = model(x)
    pred_labels = onecold(pred, 0:9)
    true_labels = onecold(y, 0:9)
    return mean(pred_labels .== true_labels)
end

println("評価関数の定義完了")

## 8. モデルの訓練

In [ ]:
# バッチサイズとエポック数の設定
batch_size = 128
epochs = 10

# データローダーの作成
train_data_batches = Flux.DataLoader((X_train, y_train), batchsize=batch_size, shuffle=true)

# 訓練ログ
train_losses = Float64[]
train_accuracies = Float64[]
test_accuracies = Float64[]

println("訓練開始...")

for epoch in 1:epochs
    epoch_loss = 0.0
    batch_count = 0
    
    for (x, y) in train_data_batches
        # 勾配の計算とパラメータの更新
        grads = Flux.gradient(params_model) do
            loss(x, y)
        end
        Flux.update!(opt, params_model, grads)
        
        epoch_loss += loss(x, y)
        batch_count += 1
    end
    
    # エポックごとの統計
    avg_loss = epoch_loss / batch_count
    train_acc = accuracy(X_train, y_train)
    test_acc = accuracy(X_test, y_test)
    
    push!(train_losses, avg_loss)
    push!(train_accuracies, train_acc)
    push!(test_accuracies, test_acc)
    
    println("Epoch $epoch/$epochs - Loss: $(round(avg_loss, digits=4)), Train Acc: $(round(train_acc*100, digits=2))%, Test Acc: $(round(test_acc*100, digits=2))%")
end

println("訓練完了!")

## 9. 訓練結果の可視化

In [ ]:
# 損失と精度のプロット
p1 = plot(1:epochs, train_losses, label="Training Loss", xlabel="Epoch", ylabel="Loss", 
          title="Training Loss", linewidth=2, marker=:circle)

p2 = plot(1:epochs, [train_accuracies test_accuracies], 
          label=["Train Accuracy" "Test Accuracy"],
          xlabel="Epoch", ylabel="Accuracy", title="Model Accuracy",
          linewidth=2, marker=:circle)

plot(p1, p2, layout=(1, 2), size=(1000, 400))

## 10. モデルの評価

In [ ]:
# 最終的なテスト精度
final_test_accuracy = accuracy(X_test, y_test)
println("最終テスト精度: $(round(final_test_accuracy*100, digits=2))%")

# 混同行列の計算
predictions = onecold(model(X_test), 0:9)
true_labels = onecold(y_test, 0:9)

# クラスごとの精度
println("\nクラスごとの精度:")
for digit in 0:9
    mask = true_labels .== digit
    class_acc = mean(predictions[mask] .== digit)
    println("数字 $digit: $(round(class_acc*100, digits=2))%")
end

## 11. 予測結果のサンプル表示

In [ ]:
# テストデータから10個のサンプルを表示
Random.seed!(42)
sample_indices = rand(1:size(X_test, 2), 10)

p = plot(layout=(2, 5), size=(1000, 500), legend=false)
for (i, idx) in enumerate(sample_indices)
    img = reshape(X_test[:, idx], 28, 28)
    true_label = onecold(y_test[:, idx], 0:9)
    pred_label = onecold(model(X_test[:, idx]), 0:9)
    
    color = pred_label == true_label ? :green : :red
    title_text = "True: $true_label, Pred: $pred_label"
    
    heatmap!(p[i], img, c=:grays, title=title_text, titlefontcolor=color, aspect_ratio=:equal)
end
display(p)

println("緑: 正解, 赤: 不正解")

## まとめ

このノートブックでは、以下を実装しました：

1. MNISTデータセットの読み込みと前処理
2. 3層のニューラルネットワークモデルの構築
3. モデルの訓練（10エポック）
4. 訓練過程の可視化
5. モデルの評価と精度の計算
6. 予測結果の可視化

Fluxを使用することで、簡潔で読みやすいコードで高性能な機械学習モデルを実装できました。